<a href="https://colab.research.google.com/github/arnavm2k3/TorchCode-Solutions/blob/main/templates/26_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🟠 Medium: LoRA (Low-Rank Adaptation)

Implement **LoRA** — parameter-efficient fine-tuning for large models.

$$h = W_0 x + \frac{\alpha}{r} B A x$$

### Signature
```python
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Requirements
- `self.linear`: frozen `nn.Linear` (weight & bias `requires_grad=False`)
- `self.lora_A`: `nn.Parameter(rank, in_features)` — random init
- `self.lora_B`: `nn.Parameter(out_features, rank)` — **zero** init
- Scaling: `alpha / rank`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.9 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [35]:
# ✏️ YOUR IMPLEMENTATION HERE

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        # frozen linear + lora_A + lora_B
        self.linear = nn.Linear(in_features, out_features)
        self.lora_A = nn.Parameter(torch.randn(rank, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))

        self.alpha = alpha
        self.rank = rank

        self.linear.weight.requires_grad_(False)
        self.linear.bias.requires_grad_(False)

    def forward(self, x):
        # base + lora
        base = self.linear(x)
        lora = (self.alpha / self.rank) * ((self.lora_B @ self.lora_A) @ x.transpose(0, 1)).transpose(0, 1)
        return base + lora

In [36]:
# 🧪 Debug
layer = LoRALinear(16, 8, rank=4)
x = torch.randn(2, 16)
print('Output:', layer(x).shape)
print('Trainable:', sum(p.numel() for p in layer.parameters() if p.requires_grad))
print('Total:    ', sum(p.numel() for p in layer.parameters()))

Output: torch.Size([2, 8])
Trainable: 96
Total:     232


In [37]:
# ✅ SUBMIT
from torch_judge import check
check('lora')


🧪 Testing: LoRA (Low-Rank Adaptation) (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Base weights frozen (1.0ms)
  ✅ [2/5] LoRA parameter shapes (0.4ms)
  ✅ [3/5] B=0 means output equals base (45.8ms)
  ✅ [4/5] Only LoRA params get gradients (21.9ms)
  ✅ [5/5] Forward computation (2.8ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (71.9ms total)
  Progress saved. Run status() to see your dashboard.

